# Plateau Search Analysis

This notebook studies two design choices of the `move_plateau` local-search operator:

1. Repeated applications of `move_plateau`
2. Different limits for accepted zero-gain moves

The analysis covers Powerlaw and Erdős-Rényi instances and distinguishes between graph size and density regime.

For solution quality, the best result across all randomized runs is used.
For runtime, the total runtime of all runs is considered.

In [5]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Repeated move_plateau analysis

This notebook analyzes the effect of applying `move_plateau` repeatedly within the same local-search pipeline.

The evaluated pipeline consists of eight consecutive applications of `move_plateau`.
The step-level results are used to compare solution quality and runtime after each application within the same randomized run.

In [6]:
EXPERIMENTS = [
    {
        "graph_type": "powerlaw",
        "dataset": "small sparse",
        "repeated_dir": Path("../results/experiment2_analysis/repeated_move/repeated_move_small_powerlaw_sparse"),
        "zero_gain_dirs": {
            "0.5V": Path("../results/experiment2_analysis/half_zeroGain/half_zeroGain_small_powerlaw_sparse"),
            "2V": Path("../results/experiment2_analysis/2V_zeroGain/2V_zeroGain_small_powerlaw_sparse"),
            "4V": Path("../results/experiment2_analysis/4V_zeroGain/4V_zeroGain_small_powerlaw_sparse"),
            "8V": Path("../results/experiment2_analysis/8V_zeroGain/8V_zeroGain_small_powerlaw_sparse"),
        },
    },
    {
        "graph_type": "powerlaw",
        "dataset": "small dense",
        "repeated_dir": Path("../results/experiment2_analysis/repeated_move/repeated_move_small_powerlaw_dense"),
        "zero_gain_dirs": {
            "0.5V": Path("../results/experiment2_analysis/half_zeroGain/half_zeroGain_small_powerlaw_dense"),
            "2V": Path("../results/experiment2_analysis/2V_zeroGain/2V_zeroGain_small_powerlaw_dense"),
            "4V": Path("../results/experiment2_analysis/4V_zeroGain/4V_zeroGain_small_powerlaw_dense"),
            "8V": Path("../results/experiment2_analysis/8V_zeroGain/8V_zeroGain_small_powerlaw_dense"),
        },
    },
    { "graph_type": "powerlaw",
      "dataset": "large sparse",
      "repeated_dir": Path("../results/experiment2_analysis/repeated_move/repeated_move_large_powerlaw_sparse"),
      "zero_gain_dirs": {
          "0.5V": Path("../results/experiment2_analysis/half_zeroGain/half_zeroGain_large_powerlaw_sparse"),
          "2V": Path("../results/experiment2_analysis/2V_zeroGain/2V_zeroGain_large_powerlaw_sparse"),
          "4V": Path("../results/experiment2_analysis/4V_zeroGain/4V_zeroGain_large_powerlaw_sparse"),
          "8V": Path("../results/experiment2_analysis/8V_zeroGain/8V_zeroGain_large_powerlaw_sparse"),
      },
    },
    { "graph_type": "powerlaw",
      "dataset": "large dense",
      "repeated_dir": Path("../results/experiment2_analysis/repeated_move/repeated_move_large_powerlaw_dense"),
      "zero_gain_dirs": {
          "0.5V": Path("../results/experiment2_analysis/half_zeroGain/half_zeroGain_large_powerlaw_dense"),
          "2V": Path("../results/experiment2_analysis/2V_zeroGain/2V_zeroGain_large_powerlaw_dense"),
          "4V": Path("../results/experiment2_analysis/4V_zeroGain/4V_zeroGain_large_powerlaw_dense"),
          "8V": Path("../results/experiment2_analysis/8V_zeroGain/8V_zeroGain_large_powerlaw_dense"),
      },
    },
    { "graph_type": "er",
      "dataset": "small sparse",
      "repeated_dir": Path("../results/experiment2_analysis/repeated_move/repeated_move_small_er_sparse"),
      "zero_gain_dirs": {
          "0.5V": Path("../results/experiment2_analysis/half_zeroGain/half_zeroGain_small_er_sparse"),
          "2V": Path("../results/experiment2_analysis/2V_zeroGain/2V_zeroGain_small_er_sparse"),
          "4V": Path("../results/experiment2_analysis/4V_zeroGain/4V_zeroGain_small_er_sparse"),
          "8V": Path("../results/experiment2_analysis/8V_zeroGain/8V_zeroGain_small_er_sparse"),
      },
    },
    { "graph_type": "er",
      "dataset": "small dense",
      "repeated_dir": Path("../results/experiment2_analysis/repeated_move/repeated_move_small_er_dense"),
      "zero_gain_dirs": {
          "0.5V": Path("../results/experiment2_analysis/half_zeroGain/half_zeroGain_small_er_dense"),
          "2V": Path("../results/experiment2_analysis/2V_zeroGain/2V_zeroGain_small_er_dense"),
          "4V": Path("../results/experiment2_analysis/4V_zeroGain/4V_zeroGain_small_er_dense"),
          "8V": Path("../results/experiment2_analysis/8V_zeroGain/8V_zeroGain_small_er_dense"),
      },
    },
    { "graph_type": "er",
      "dataset": "large sparse",
      "repeated_dir": Path("../results/experiment2_analysis/repeated_move/repeated_move_large_er_sparse"),
      "zero_gain_dirs": {
          "0.5V": Path("../results/experiment2_analysis/half_zeroGain/half_zeroGain_large_er_sparse"),
          "2V": Path("../results/experiment2_analysis/2V_zeroGain/2V_zeroGain_large_er_sparse"),
          "4V": Path("../results/experiment2_analysis/4V_zeroGain/4V_zeroGain_large_er_sparse"),
          "8V": Path("../results/experiment2_analysis/8V_zeroGain/8V_zeroGain_large_er_sparse"),
      },
    },
    { "graph_type": "er",
      "dataset": "large dense",
      "repeated_dir": Path("../results/experiment2_analysis/repeated_move/repeated_move_large_er_dense"),
      "zero_gain_dirs": {
          "0.5V": Path("../results/experiment2_analysis/half_zeroGain/half_zeroGain_large_er_dense"),
          "2V": Path("../results/experiment2_analysis/2V_zeroGain/2V_zeroGain_large_er_dense"),
          "4V": Path("../results/experiment2_analysis/4V_zeroGain/4V_zeroGain_large_er_dense"),
          "8V": Path("../results/experiment2_analysis/8V_zeroGain/8V_zeroGain_large_er_dense"),
      },
    },
]

In [8]:
REPEATED_PIPELINE = ",".join(["move_plateau"] * 8)

STEP_LABELS = {
    step_index: f"After step {step_index + 1}"
    for step_index in range(8)
}

STEP_ORDER = [
    f"After step {step_number + 1}"
    for step_number in range(8)
]

REPEATED_PIPELINE

'move_plateau,move_plateau,move_plateau,move_plateau,move_plateau,move_plateau,move_plateau,move_plateau'

In [ ]:
sample_path = EXPERIMENTS[0]["repeated_dir"] / "step_results.csv"

sample_steps = pd.read_csv(sample_path)

print("Shape:", sample_steps.shape)
print("\nColumns:")
print(sample_steps.columns.tolist())

print("\nAvailable pipelines:")
for pipeline in sample_steps["pipeline"].drop_duplicates():
    print(pipeline)

print("\nAvailable step indices:")
print(sorted(sample_steps["step_index"].dropna().unique()))

print("\nStep-index dtype:")
print(sample_steps["step_index"].dtype)

In [ ]:
sample_repeated_steps = sample_steps[
    sample_steps["pipeline"] == REPEATED_PIPELINE
    ].copy()

print("Filtered shape:", sample_repeated_steps.shape)
print(
    "Step indices:",
    sorted(sample_repeated_steps["step_index"].unique()),
)

In [ ]:
RUN_KEY = [
    "dataset",
    "instance",
    "start_partition",
    "run",
    "seed",
    "pipeline",
]

steps_per_run = (
    sample_repeated_steps
    .groupby(RUN_KEY, observed=True)
    .size()
    .rename("num_steps")
    .reset_index()
)

steps_per_run["num_steps"].value_counts().sort_index()

In [ ]:
incomplete_runs = steps_per_run[
    steps_per_run["num_steps"] != 8
    ]

print("Incomplete runs:", len(incomplete_runs))
incomplete_runs.head()

In [ ]:
sample_steps.shape
sample_steps.columns.tolist()
sample_steps["pipeline"].drop_duplicates().tolist()
steps_per_run["num_steps"].value_counts()